In [1]:
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    pipeline,
    BitsAndBytesConfig
)

from peft import PeftModel
import torch 

In [2]:
checkpoint = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

In [3]:
base_model = AutoModelForCausalLM.from_pretrained(
    checkpoint,
    quantization_config=bnb,
    device_map="auto")

In [4]:
tokenizer = AutoTokenizer.from_pretrained(
    "finance_chat_model"
)  

In [5]:
model = PeftModel.from_pretrained(
    base_model,
    "finance_chat_model"
) 

In [6]:
generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer
) 

Device set to use cuda:0


In [7]:
fine_tuned_generator=pipeline("text-generation",model=model,tokenizer=tokenizer)
base_generator=pipeline("text-generation",model=checkpoint,tokenizer=tokenizer)  

Device set to use cuda:0
Device set to use cuda:0


In [8]:
with open("model_comparison.txt", "w", encoding="utf-8") as f:

    questions = [
        "What is asset allocation in a portfolio?.",
        "How can rising interest rates affect bond prices?",
        "Explain the benefits of investing through SIPs",
        "A person wants passive income from investments. What options exist?",
        "Compare SIP and lump sum investing for a beginner."]
        

    for q in questions:

        # prompt = f"<s>[INST] {q} [/INST]"
        prompt = f"Question: {q}\nAnswer:"

        print("\n" + "=" * 80)
        print("QUESTION:", q)

        # Base Model
        base_result = base_generator(
            prompt,
            max_new_tokens=150,
            repetition_penalty=1.2,
            do_sample=False,
            eos_token_id=tokenizer.eos_token_id,
            return_full_text=False
        )

        # Fine-Tuned Model
        fine_tuned_result = fine_tuned_generator(
            prompt,
            max_new_tokens=150,
            repetition_penalty=1.3,
            eos_token_id=tokenizer.eos_token_id,
            do_sample=False,
            return_full_text=False
        )

        print("\nBASE MODEL:")
        print(base_result[0]["generated_text"])

        print("\nFINE-TUNED MODEL:")
        print(fine_tuned_result[0]["generated_text"])

        # Save to file
        f.write("\n" + "=" * 80 + "\n")
        f.write(f"QUESTION: {q}\n\n")

        f.write("BASE MODEL:\n")
        f.write(base_result[0]["generated_text"])

        f.write("\n\nFINE-TUNED MODEL:\n")
        f.write(fine_tuned_result[0]["generated_text"])

        f.write("\n\n")

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



QUESTION: What is asset allocation in a portfolio?.

BASE MODEL:
 Asset allocation refers to the distribution of assets among different investment classes, such as stocks, bonds, and cash. It involves selecting an appropriate mix of these assets based on various factors like risk tolerance, time horizon, and expected return. The goal is to achieve diversification across different asset classes while minimizing overall risk.

FINE-TUNED MODEL:
 Asset Allocation refers to the proportion of assets that an investor allocates towards different types of securities. The goal of this approach is to achieve diversification, which means reducing risk by spreading out your investments across various markets and industries. Investors who follow this strategy are known as diversifiers or diversified investors. Diversifying can be done through stocks (domestic), bonds (foreign) and cash reserves. A good example would be Warren Buffet's Berkshire Hathaway Company. They have 10% equity exposure with 